# 目的
* 当該ファイルでは，日本語学術論文データベースを対象とした全文検索ベースの書誌同定手法(以降，本手法)を実行する．
* 全文検索によるスコアとして，Reference Coverage(CC)・Candidate Coverage(CC)・MC(Mutual Coverage)の3つを用いる．
* 参考文献文字列1件に対する出力を確認することを想定している．

## 要実行

In [2]:
# モジュールの読み込み
from pathlib import Path
import sys
import pysolr

TOOLS_DIR = Path.cwd().resolve().parent / "8プロジェクト関連" / "8-3共有プログラム"
if str(TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(TOOLS_DIR))

import wakati
import generate_query

# 接続文字列
solr_url = "http://localhost:8983/solr/tanigawa_jalc"

# pysolrのクライアントの初期化
solr = pysolr.Solr(solr_url,timeout=10)

## 実行

### Reference Coverage(RC)

In [20]:
# リスト型データの各要素をトークン化して結合
def list_to_token(lis):
    tokens = []
    for val in lis:
        tokens += wakati.get_token(val)
    return tokens

# sim_r 算出用の候補文献のトークン集合を得る
def get_sim_r_token(document):
    tokens = []
    # トークン化するフィールド一覧
    field_names = ["creator","title","journal","issued","volume_issue","page_range"]
    # 各フィールドをトークン化して結合
    for field_name in field_names:
        tokens += list(set(list_to_token(document[field_name])))    
    return tokens    

# sim_r の算出
def calc_sim_r(reference_token,sim_r_token):
    # sim_r スコア
    sim_r_score = 0
    # 両方のトークン集合の共通要素の数
    common_token_num = 0

    for token in reference_token:
        if str(token) in sim_r_token:
            common_token_num += 1
        """    
        else:
            print(f"含まれなかったトークン：{token}")
        """        

    sim_r_score = common_token_num / len(reference_token)

    return sim_r_score            

# sim_rの閾値
sim_r_threshold = 0.9375

# クエリ（参考文献文字列）
query = "吉本葵，林貴宏，金沢輝一，平井克之，書誌同定における同一著作問題の解決へ向けたグラフデータベースの活用，情報処理学会全国大会論文集，5N-05, 分冊1, p.497-498, 2025.3"

# 検索オプション
search_options = {
    'q.op':'OR',
    'fl':'doi,first_author,creator,title,journal,issued,volume_issue,page_range,score',
    'df':'jalcdata',
    'rows':10
}

# クエリのエスケープ処理
escape_query = generate_query.escape_solr_query(query)

# 参考文献文字列のトークナイズ
query_token = list(set(wakati.get_token(query)))

# solrによる全文検索
results = solr.search(escape_query,**search_options)

# リランキング処理
max_sim_r_score = -1
candidate_document = "" # 最大スコアの候補文献
for result in results:
    # 候補文献のトークン化
    sim_r_token = get_sim_r_token(result)
    # sim_rの算出
    sim_r_score = calc_sim_r(query_token,sim_r_token) 
    if max_sim_r_score < sim_r_score:
        candidate_document = result
        max_sim_r_score = sim_r_score

print("---実行結果---")
print(f"クエリ = {query}")
# 閾値評価
if max_sim_r_score >= sim_r_threshold:
    print("▼同定された文献情報")  
    print(f"DOI        | {candidate_document["doi"]}")
    print(f"著者名　　　 | {candidate_document["creator"]}")
    print(f"論文タイトル | {candidate_document["title"]}")
    print(f"雑誌名　　　 | {candidate_document["journal"]}")
    print(f"出版年　　　 | {candidate_document["issued"]}")
    print(f"巻号　　　　 | {candidate_document["volume_issue"]}")
    print(f"ページ番号　 | {candidate_document["page_range"]}")
    print("-"*20)
    print(f"閾値評価：RC = {max_sim_r_score} (閾値 = {sim_r_threshold:.4f})")
else:
    print("▼同定された文献なし")
    print("-"*20)
    print("候補文献のDOI | " + candidate_document["doi"])
    print(f"閾値評価：RC = {max_sim_r_score} (閾値 = {sim_r_threshold:.4f})")                

---実行結果---
クエリ = 吉本葵，林貴宏，金沢輝一，平井克之，書誌同定における同一著作問題の解決へ向けたグラフデータベースの活用，情報処理学会全国大会論文集，5N-05, 分冊1, p.497-498, 2025.3
▼同定された文献なし
--------------------
候補文献のDOI | 10.11517/pjsai.JSAI2015.0_1K5NFC05b3
閾値評価：RC = 0.2857142857142857 (閾値 = 0.9375)


### Candidate Coverage(CC)

In [21]:
# リスト型データの各要素のトークン化
def list_to_token(lis):
    tokens = []
    for val in lis:
        tokens.append(wakati.get_token(val))
    return tokens

# sim_p 算出用のトークン集合を得る
def get_sim_p_token(document):
    sim_p_token = []
    # トークン化するフィールド名一覧
    field_names = ["first_author","title","journal","issued","volume_issue","page_range"]
    # 各フィールドをトークン化
    for field in field_names:
        sim_p_token.append(list_to_token(document[field]))
    return sim_p_token

# sim_p 算出関数
def calc_sim_p(reference_token,sim_p_token):
    sim_p_score = 0
    for i in range(len(sim_p_token)):
        max_score = 0
        for j in range(len(sim_p_token[i])):
            common_token_num = 0
            for token in sim_p_token[i][j]:
                if token in reference_token:
                    common_token_num += 1
            tmp = common_token_num / len(sim_p_token[i][j])
            if tmp > max_score:
                max_score = tmp
        sim_p_score += max_score
    return sim_p_score / 6                   

# 空のリストを削除
def remove_empty_lists(data):
    if isinstance(data, list):
        # リストであれば、各要素に対して再帰的に処理
        return [remove_empty_lists(item) for item in data if remove_empty_lists(item)]
    else:
        # リストでなければ、そのまま返す
        return data

# sim_pの閾値
sim_p_threshold = 0.8730

# クエリ
query = "吉本葵，林貴宏，金沢輝一，平井克之，書誌同定における同一著作問題の解決へ向けたグラフデータベースの活用，情報処理学会全国大会論文集，5N-05, 分冊1, p.497-498, 2025.3"

# 検索オプション
search_options = {
    'q.op':'OR',
    'fl':'doi,first_author,creator,title,journal,issued,volume_issue,page_range,score',
    'df':'jalcdata',
    'rows':10
}

# クエリのエスケープ処理
escape_query = generate_query.escape_solr_query(query)

# 参考文献文字列のトークナイズ
query_token = list(set(wakati.get_token(query)))

# solrによる全文検索
results = solr.search(escape_query,**search_options)

# リランキング処理
max_sim_p_score = -1
candidate_document = "" # 最大スコアの候補文献
for result in results:
    # 候補文献のトークン化
    sim_p_token = get_sim_p_token(result)
    # 空のリストを除去
    sim_p_token = remove_empty_lists(sim_p_token)
    # sim_p の算出
    sim_p_score = calc_sim_p(query_token,sim_p_token)
    if max_sim_p_score < sim_p_score:
        candidate_document = result
        max_sim_p_score = sim_p_score

print("---実行結果---")
print(f"クエリ = {query}")
# 閾値評価
if max_sim_p_score >= sim_p_threshold:
    print("▼同定された文献情報")  
    print(f"DOI        | {candidate_document["doi"]}")
    print(f"著者名　　　 | {candidate_document["creator"]}")
    print(f"論文タイトル | {candidate_document["title"]}")
    print(f"雑誌名　　　 | {candidate_document["journal"]}")
    print(f"出版年　　　 | {candidate_document["issued"]}")
    print(f"巻号　　　　 | {candidate_document["volume_issue"]}")
    print(f"ページ番号　 | {candidate_document["page_range"]}")
    print("-"*20)
    print(f"閾値評価：CC = {max_sim_p_score} (閾値 = {sim_p_threshold:.4f})")
else:
    print("▼同定された文献なし")
    print("-"*20)
    print("候補文献のDOI | " + candidate_document["doi"])
    print(f"閾値評価：CC = {max_sim_p_score} (閾値 = {sim_p_threshold:.4f})")            

---実行結果---
クエリ = 吉本葵，林貴宏，金沢輝一，平井克之，書誌同定における同一著作問題の解決へ向けたグラフデータベースの活用，情報処理学会全国大会論文集，5N-05, 分冊1, p.497-498, 2025.3
▼同定された文献なし
--------------------
候補文献のDOI | 10.11517/pjsai.JSAI2015.0_1K5NFC05b3
閾値評価：CC = 0.2786324786324786 (閾値 = 0.8730)


### Mutual Coverage(MC)

In [23]:
# リスト型データの各要素をトークン化して結合(sim_r)
def list_to_token_sim_r(lis):
    tokens = []
    for val in lis:
        tokens += wakati.get_token(val)
    return tokens

# リスト型データの各要素のトークン化(sim_p)
def list_to_token_sim_p(lis):
    tokens = []
    for val in lis:
        tokens.append(wakati.get_token(val))
    return tokens

# sim_r 算出用の候補文献のトークン集合を得る
def get_sim_r_token(document):
    tokens = []
    # トークン化するフィールド一覧
    field_names = ["creator","title","journal","issued","volume_issue","page_range"]
    # 各フィールドをトークン化して結合
    for field_name in field_names:
        tokens += list(set(list_to_token_sim_r(document[field_name])))    
    return tokens    

# sim_r の算出
def calc_sim_r(reference_token,sim_r_token):
    # sim_r スコア
    sim_r_score = 0
    # 両方のトークン集合の共通要素の数
    common_token_num = 0

    for token in reference_token:
        if str(token) in sim_r_token:
            common_token_num += 1
        """    
        else:
            print(f"含まれなかったトークン：{token}")
        """        

    sim_r_score = common_token_num / len(reference_token)

    return sim_r_score            

# sim_p 算出用のトークン集合を得る
def get_sim_p_token(document):
    sim_p_token = []
    # トークン化するフィールド名一覧
    field_names = ["first_author","title","journal","issued","volume_issue","page_range"]
    # 各フィールドをトークン化
    for field in field_names:
        sim_p_token.append(list_to_token_sim_p(document[field]))
    return sim_p_token

# sim_p 算出関数
def calc_sim_p(reference_token,sim_p_token):
    sim_p_score = 0
    for i in range(len(sim_p_token)):
        max_score = 0
        for j in range(len(sim_p_token[i])):
            common_token_num = 0
            for token in sim_p_token[i][j]:
                if token in reference_token:
                    common_token_num += 1
            tmp = common_token_num / len(sim_p_token[i][j])
            if tmp > max_score:
                max_score = tmp
        sim_p_score += max_score
    return sim_p_score / 6                   

# 空のリストを削除
def remove_empty_lists(data):
    if isinstance(data, list):
        # リストであれば、各要素に対して再帰的に処理
        return [remove_empty_lists(item) for item in data if remove_empty_lists(item)]
    else:
        # リストでなければ、そのまま返す
        return data

# sim_rの閾値
sim_r_threshold = 0.9375

# sim_pの閾値
sim_p_threshold = 0.8730

# クエリ
query = "吉本葵，林貴宏，金沢輝一，平井克之，書誌同定における同一著作問題の解決へ向けたグラフデータベースの活用，情報処理学会全国大会論文集，5N-05, 分冊1, p.497-498, 2025.3"

# 検索オプション
search_options = {
    'q.op':'OR',
    'fl':'doi,first_author,creator,title,journal,issued,volume_issue,page_range,score',
    'df':'jalcdata',
    'rows':10
}

# クエリのエスケープ処理
escape_query = generate_query.escape_solr_query(query)

# 参考文献文字列のトークナイズ
query_token = list(set(wakati.get_token(query)))

# solrによる全文検索
results = solr.search(escape_query,**search_options)

# リランキング処理
max_sim_p_score = -1
max_sim_r_score = -1
candidate_document = "" # 候補文献
sum_sim_r_sim_p = -1
for result in results:
    # 候補文献のトークン化
    sim_r_token = get_sim_r_token(result)
    sim_p_token = get_sim_p_token(result)
    # 空のリストを除去
    sim_p_token = remove_empty_lists(sim_p_token)
    # 類似度の算出
    sim_r_score,sim_p_score = calc_sim_r(query_token,sim_r_token),calc_sim_p(query_token,sim_p_token)
    if sum_sim_r_sim_p < sim_r_score + sim_p_score:
        candidate_document = result
        max_sim_r_score = sim_r_score
        max_sim_p_score = sim_p_score
        sum_sim_r_sim_p = sim_r_score + sim_p_score

print("---実行結果---")
print(f"クエリ = {query}")
# 閾値評価
if max_sim_p_score >= sim_p_threshold and max_sim_r_score >= sim_r_threshold:
    print("▼同定された文献情報")  
    print(f"DOI        | {candidate_document["doi"]}")
    print(f"著者名　　　 | {candidate_document["creator"]}")
    print(f"論文タイトル | {candidate_document["title"]}")
    print(f"雑誌名　　　 | {candidate_document["journal"]}")
    print(f"出版年　　　 | {candidate_document["issued"]}")
    print(f"巻号　　　　 | {candidate_document["volume_issue"]}")
    print(f"ページ番号　 | {candidate_document["page_range"]}")
    print("-"*20)
    print(f"閾値評価：RC = {max_sim_r_score} (閾値 = {sim_r_threshold:.4f})")
    print(f"閾値評価：CC = {max_sim_p_score} (閾値 = {sim_p_threshold:.4f})")
else:
    print("▼同定された文献なし")
    print("-"*20)
    print("候補文献のDOI | " + candidate_document["doi"])
    print(f"閾値評価：RC = {max_sim_r_score} (閾値 = {sim_r_threshold:.4f})")
    print(f"閾値評価：CC = {max_sim_p_score} (閾値 = {sim_p_threshold:.4f})")         

---実行結果---
クエリ = 吉本葵，林貴宏，金沢輝一，平井克之，書誌同定における同一著作問題の解決へ向けたグラフデータベースの活用，情報処理学会全国大会論文集，5N-05, 分冊1, p.497-498, 2025.3
▼同定された文献なし
--------------------
候補文献のDOI | 10.11517/pjsai.JSAI2015.0_1K5NFC05b3
閾値評価：RC = 0.2857142857142857 (閾値 = 0.9375)
閾値評価：CC = 0.2786324786324786 (閾値 = 0.8730)
